# Homework 4 — Training Notebook

**Setup:** In Colab go to `Runtime → Change runtime type → T4 GPU`, then run every cell top-to-bottom.

No Google Drive needed. Code is uploaded directly to the Colab runtime; the dataset is downloaded from the course URL.

---

### Colab disconnected / kernel restarted?

**Run only the "Quick Reconnect" cell below** — you do NOT need to re-upload code or re-download data. Files in `/content/` survive idle reconnects within the same session. Only re-run earlier cells if a ✗ appears next to something you need.

---

### Preventing idle disconnects

Paste this into the **browser console** (`Ctrl+Shift+J` / `Cmd+Option+J`) to keep the session alive:

```javascript
setInterval(() => document.querySelector('colab-connect-button')?.click(), 60000)
```

## Step 0 — Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU. Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
## ─── QUICK RECONNECT — run this first after any disconnect ───────────────────
import os, sys
from pathlib import Path

PROJECT = '/content/homework4'
os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

# Status check — shows what's already done so you know where to resume
checks = {
    'Code uploaded':       Path('homework/finetune.py').exists(),
    'Dataset downloaded':  Path('data/train').exists() and any(Path('data/train').glob('*_info.json')),
    'QA pairs generated':  Path('data/train/generated_qa_pairs.json').exists(),
    'Captions generated':  Path('data/train/generated_captions.json').exists(),
    'VLM checkpoint':      Path('homework/vlm_model/adapter_config.json').exists(),
    'CLIP checkpoint':     Path('homework/clip_model/adapter_config.json').exists(),
}
for label, ok in checks.items():
    print(f"{'✓' if ok else '✗'}  {label}")
print('\nWorking directory:', os.getcwd())

## Step 1 — Upload Your Code

Run the cell below. A file-picker will appear — select the **`homework_code.zip`** you created locally:

```bash
# Run this on your local machine first to create the zip:
cd /path/to/homework4_aug_4
zip -r homework_code.zip homework/ requirements.txt bundle.py grader/ --exclude "*.pyc" "*/__pycache__/*"
```

In [ ]:
from google.colab import files
import zipfile, os, sys, site
from pathlib import Path

uploaded = files.upload()   # pick homework_code.zip
zip_name = list(uploaded.keys())[0]

PROJECT = '/content/homework4'
os.makedirs(PROJECT, exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(PROJECT)

os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

# Register path permanently so it survives kernel restarts in this session
pth = Path(site.getsitepackages()[0]) / 'homework4.pth'
pth.write_text(PROJECT + '\n')
print(f'Path registered: {pth}')

print('Working directory:', os.getcwd())
print('Contents:', os.listdir('.'))

## Step 2 — Install Dependencies

Colab already has `torch` and `torchvision`. We only install the extras.

The cell below will **automatically restart the runtime** after installation. Once it restarts, run only the **Quick Reconnect** cell — your uploaded code and downloaded data are still in `/content/homework4`.

In [ ]:
%pip install -q transformers==4.52.4 peft==0.16.0 fire tqdm opencv-python tensorboard

# Restart the runtime so the newly installed packages are fully loaded.
# After restart, run ONLY the Quick Reconnect cell — you can skip Steps 1 & 2.
print('Restarting runtime to apply package updates...')
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

## Step 4 — Download Dataset

In [ ]:
from pathlib import Path
import os

data_dir = Path('data/train')
if not data_dir.exists() or not any(data_dir.glob('*_info.json')):
    print('Downloading dataset...')
    os.system('wget -q https://utexas.box.com/shared/static/qubjm5isldqvyimfj9rsmbnvnbezwcv4.zip -O supertux_data.zip')
    os.system('unzip -q supertux_data.zip')
    os.remove('supertux_data.zip')
    print('Done.')
else:
    n = len(list(data_dir.glob('*_info.json')))
    print(f'Dataset already present ({n} info files in data/train)')

---
## Part 1 — VLM Fine-Tuning

Target: **≥ 70% accuracy** on the validation set (50 pts).

### 1a — Generate QA Training Data

In [ ]:
import json, random
from pathlib import Path
from homework.generate_qa import generate_qa_pairs

data_train = Path('data/train')
out_file = data_train / 'generated_qa_pairs.json'

if out_file.exists():
    print(f'QA file already exists: {out_file}')
else:
    # Build image name set once — avoids O(n²) directory scans in the loop
    existing_images = {f.name for f in data_train.glob('*_im.jpg')}
    print(f'Found {len(existing_images)} images, generating QA pairs...')

    all_qa = []
    for info_file in sorted(data_train.glob('*_info.json')):
        base = info_file.stem.replace('_info', '')
        with open(info_file) as f:
            info = json.load(f)
        num_views = len(info.get('detections', []))
        for view_index in range(num_views):
            img_name = f'{base}_{view_index:02d}_im.jpg'
            if img_name not in existing_images:
                continue
            try:
                pairs = generate_qa_pairs(str(info_file), view_index)
                for p in pairs:
                    p['image_file'] = f'train/{img_name}'
                all_qa.extend(pairs)
            except Exception:
                pass

    random.shuffle(all_qa)
    out_file.write_text(json.dumps(all_qa, indent=2))
    print(f'Generated {len(all_qa)} QA pairs → {out_file}')

### 1b — Train VLM

In [ ]:
import gc, os
import torch
from pathlib import Path
from peft import LoraConfig, TaskType, get_peft_model
from transformers import Trainer, TrainerCallback, TrainingArguments

from homework.base_vlm import DEVICE, BaseVLM
from homework.data import VQADataset, benchmark
from homework.finetune import VQADatasetForTraining, custom_data_collator, processor

# ── Clear any leftover GPU state from previous runs ────────────────────────
for _var in ['model', 'trainer', 'vlm', 'train_dataset']:
    if _var in dir():
        del _var
gc.collect()
torch.cuda.empty_cache()
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
print(f'GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB / '
      f'{torch.cuda.mem_get_info()[1]/1e9:.1f} GB total')

# ── Early-stopping callback ────────────────────────────────────────────────
class EarlyStopOnAccuracy(TrainerCallback):
    """Evaluates on valid_grader every eval_steps and stops when target_acc is met."""

    def __init__(self, vlm, target_acc=0.75, eval_steps=100, max_eval_samples=128):
        self.vlm              = vlm
        self.target_acc       = target_acc
        self.eval_steps       = eval_steps
        self.max_eval_samples = max_eval_samples
        self._eval_dataset    = None

    @property
    def eval_dataset(self):
        if self._eval_dataset is None:
            self._eval_dataset = VQADataset('valid_grader')
        return self._eval_dataset

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step == 0 or state.global_step % self.eval_steps != 0:
            return control

        self.vlm.model = model
        model.eval()
        result = benchmark(self.vlm, self.eval_dataset, self.max_eval_samples)
        print(f'\nStep {state.global_step} | val accuracy: {result.accuracy:.1%} | target: {self.target_acc:.0%}')
        model.train()

        if result.accuracy >= self.target_acc:
            print(f'Target {self.target_acc:.0%} reached at step {state.global_step} — stopping early.')
            control.should_training_stop = True

        return control

# ── Setup ──────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path('homework/vlm_model')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

vlm   = BaseVLM()
model = get_peft_model(
    vlm.model,
    LoraConfig(
        task_type=TaskType.CAUSAL_LM, inference_mode=False,
        r=8, lora_alpha=32, lora_dropout=0.0,
        target_modules='all-linear', bias='none',
    ),
)
model.print_trainable_parameters()
model.config.use_cache = False
model.enable_input_require_grads()
model.train()

train_dataset = VQADatasetForTraining(VQADataset('train'), processor)

training_args = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR),
    logging_dir                 = str(OUTPUT_DIR),
    report_to                   = 'tensorboard',
    num_train_epochs            = 1,
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 8,
    gradient_checkpointing      = True,
    learning_rate               = 2e-4,        # halved — 5e-4 caused mode collapse
    warmup_ratio                = 0.05,        # gentle ramp-up to avoid early collapse
    weight_decay                = 0.01,        # regularisation
    lr_scheduler_type           = 'cosine',
    bf16                        = DEVICE == 'cuda',
    logging_steps               = 1,
    save_strategy               = 'steps',
    save_steps                  = 50,
    save_total_limit            = 2,
    label_names                 = ['labels'],
    dataloader_num_workers      = 2,
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_dataset,
    data_collator = custom_data_collator,
    callbacks     = [EarlyStopOnAccuracy(vlm, target_acc=0.75, eval_steps=100, max_eval_samples=128)],
)

# ── Start fresh (collapsed checkpoints are not worth resuming) ────────────
print('Starting fresh — previous run collapsed, not resuming from checkpoint')
trainer.train()
trainer.save_model(str(OUTPUT_DIR))
print(f'\nVLM saved to {OUTPUT_DIR}')

### 1c — Evaluate VLM

In [ ]:
from homework.finetune import load as vlm_load
from homework.data import VQADataset, benchmark

vlm     = vlm_load('vlm_model')
testset = VQADataset('valid_grader')
result  = benchmark(vlm, testset, 128)

print()
print('=' * 45)
print(f'  VLM Accuracy : {result.accuracy:.1%}')
print(f'  Correct      : {int(result.accuracy * len(result.samples))} / {len(result.samples)}')
print(f'  Target       : 70.0%')
print(f'  Status       : {"✓  PASS — ready to submit" if result.accuracy >= 0.70 else "✗  below target — keep training"}')
print('=' * 45)

---
## Part 2 — CLIP

Target: **≥ 70% accuracy** on multi-choice QA (50 pts, random baseline is 20%).

### 2a — Generate Caption Training Data

In [ ]:
import json, random
from pathlib import Path
from homework.generate_captions import generate_caption

data_train = Path('data/train')
out_file = data_train / 'generated_captions.json'

if out_file.exists():
    print(f'Captions file already exists: {out_file}')
else:
    # Build image name set once — avoids O(n²) directory scans in the loop
    existing_images = {f.name for f in data_train.glob('*_im.jpg')}
    print(f'Found {len(existing_images)} images, generating captions...')

    all_caps = []
    for info_file in sorted(data_train.glob('*_info.json')):
        base = info_file.stem.replace('_info', '')
        with open(info_file) as f:
            info = json.load(f)
        num_views = len(info.get('detections', []))
        for view_index in range(num_views):
            img_name = f'{base}_{view_index:02d}_im.jpg'
            if img_name not in existing_images:
                continue
            try:
                for cap in generate_caption(str(info_file), view_index):
                    all_caps.append({
                        'image_file': f'train/{img_name}',
                        'caption': cap,
                    })
            except Exception:
                pass

    random.shuffle(all_caps)
    out_file.write_text(json.dumps(all_caps, indent=2))
    print(f'Generated {len(all_caps)} captions → {out_file}')

### 2b — Train CLIP

In [ ]:
import gc, os
import torch
import torchvision as tv
from pathlib import Path
from PIL import Image
from peft import LoraConfig, TaskType, get_peft_model
from transformers import Trainer, TrainerCallback, TrainingArguments

from homework.base_vlm import DEVICE as device, BaseVLM
from homework.clip import (
    CLIP, CaptionDatasetForTraining, clip_data_collator,
    compute_clip_loss, get_target_modules_for_lora, processor,
)
from homework.data import CaptionDataset, MultiChoiceQADataset

# ── Clear GPU state ────────────────────────────────────────────────────────
for _var in ['model', 'trainer', 'vlm']:
    if _var in dir():
        del _var
gc.collect()
torch.cuda.empty_cache()
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
print(f'GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB / '
      f'{torch.cuda.mem_get_info()[1]/1e9:.1f} GB total')

# ── Early-stopping callback ────────────────────────────────────────────────
class ClipEarlyStop(TrainerCallback):
    """Evaluates CLIP on valid_grader every eval_steps; stops when target_acc is met."""

    def __init__(self, target_acc=0.75, eval_steps=50, max_eval_samples=256):
        self.target_acc       = target_acc
        self.eval_steps       = eval_steps
        self.max_eval_samples = max_eval_samples
        self._testset         = None
        self._image_proc      = tv.transforms.Compose([
            tv.transforms.Resize(192),
            tv.transforms.CenterCrop(192),
            tv.transforms.ToTensor(),
            tv.transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ])

    @property
    def testset(self):
        if self._testset is None:
            self._testset = MultiChoiceQADataset('valid_grader')
        return self._testset

    def on_step_end(self, args, state, control, model=None, **kwargs):
        if state.global_step == 0 or state.global_step % self.eval_steps != 0:
            return control

        clip_model = model.model  # unwrap PeftModel → CLIP
        clip_model.eval()
        correct = total = 0

        with torch.no_grad():
            for i, pair in enumerate(self.testset):
                if i >= self.max_eval_samples:
                    break
                image = Image.open(pair['image_path']).convert('RGB')
                pv = self._image_proc(image).unsqueeze(0).to(device)
                if device == 'cuda':
                    pv = pv.bfloat16()
                ti = processor(
                    text=[s + processor.tokenizer.eos_token for s in pair['candidates']],
                    return_tensors='pt', padding=True, truncation=True,
                )
                ids  = ti['input_ids'].long().to(device)
                mask = ti['attention_mask'].to(device)
                vf, tf, _ = clip_model(pv, ids, mask)
                if torch.matmul(vf, tf.T).argmax(dim=-1).item() == pair['correct_index']:
                    correct += 1
                total += 1

        acc = correct / total if total else 0.0
        print(f'\nStep {state.global_step} | CLIP val accuracy: {acc:.1%} | target: {self.target_acc:.0%}')
        clip_model.train()

        if acc >= self.target_acc:
            print(f'Target {self.target_acc:.0%} reached at step {state.global_step} — stopping early.')
            control.should_training_stop = True

        return control

# ── Setup ──────────────────────────────────────────────────────────────────
OUTPUT_DIR = Path('homework/clip_model')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

vlm            = BaseVLM()
vision_encoder = vlm.model.model.vision_model
text_encoder   = vlm.model.model.text_model
model          = CLIP(vision_encoder, text_encoder).to(device)
if device == 'cuda':
    model = model.bfloat16()
model.set_trainable_parameters()

peft_config = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION, inference_mode=False,
    r=8, lora_alpha=32, lora_dropout=0.0,
    target_modules=get_target_modules_for_lora(model),
    bias='none',
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
model.to(device)
model.train()
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

train_dataset = CaptionDatasetForTraining(CaptionDataset('train'), processor)

training_args = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR),
    logging_dir                 = str(OUTPUT_DIR),
    report_to                   = 'tensorboard',
    num_train_epochs            = 3,
    per_device_train_batch_size = 256,
    gradient_accumulation_steps = 1,
    gradient_checkpointing      = True,
    learning_rate               = 5e-4,
    bf16                        = device == 'cuda',
    logging_steps               = 1,
    save_strategy               = 'steps',
    save_steps                  = 50,
    save_total_limit            = 2,
    label_names                 = ['labels'],
    dataloader_num_workers      = 2,
)

trainer = Trainer(
    model             = model,
    args              = training_args,
    train_dataset     = train_dataset,
    data_collator     = clip_data_collator,
    compute_loss_func = compute_clip_loss,
    callbacks         = [ClipEarlyStop(target_acc=0.75, eval_steps=50, max_eval_samples=256)],
)

# ── Auto-resume from latest checkpoint if one exists ──────────────────────
_checkpoints = sorted(OUTPUT_DIR.glob('checkpoint-*'), key=lambda x: int(x.name.split('-')[1]))
_resume = str(_checkpoints[-1]) if _checkpoints else None
print(f'Resuming from: {_resume}' if _resume else 'Starting fresh — no checkpoint found')

trainer.train(resume_from_checkpoint=_resume)

# Save model + projection head weights
trainer.save_model(str(OUTPUT_DIR))
model.model.save_pretrained(str(OUTPUT_DIR))
print(f'\nCLIP saved to {OUTPUT_DIR}')

### 2c — Evaluate CLIP

In [ ]:
from homework.clip import test as clip_test

clip_test('clip_model')

---
## Submission

Replace `UT_ID` with your actual UT EID, then run the cell. It will:
1. Strip intermediate checkpoints to stay under the 50 MB limit
2. Build the submission zip with `bundle.py`
3. Run the local grader to verify
4. Download the zip to your machine

In [ ]:
import os, shutil, subprocess
from pathlib import Path
from google.colab import files

UT_ID = 'ja56388'   # ← your UT EID

homework = Path('homework')

# ── Remove checkpoint folders ─────────────────────────────────────────────
for ckpt in homework.rglob('checkpoint-*'):
    shutil.rmtree(ckpt, ignore_errors=True)
    print(f'Removed {ckpt}')

# ── Remove demo_train (not part of submission) ────────────────────────────
demo = homework / 'demo_train'
if demo.exists():
    shutil.rmtree(demo)
    print(f'Removed {demo}')

# ── Remove TensorBoard event files ────────────────────────────────────────
for f in homework.rglob('events.out.tfevents.*'):
    f.unlink()
    print(f'Removed {f}')
for tb_dir in homework.rglob('tensorboard'):
    shutil.rmtree(tb_dir, ignore_errors=True)
    print(f'Removed {tb_dir}')

# ── Remove demo_clip if it exists ─────────────────────────────────────────
demo_clip = homework / 'demo_clip'
if demo_clip.exists():
    shutil.rmtree(demo_clip)
    print(f'Removed {demo_clip}')

# ── Upload bundle.py + grader/ if they weren't in the original zip ────────
if not Path('bundle.py').exists():
    print('bundle.py not found — upload it now')
    uploaded = files.upload()
    for name, data in uploaded.items():
        Path(name).write_bytes(data)

if not Path('grader').exists():
    print('grader/ not found — upload grader.zip')
    uploaded = files.upload()
    for name, data in uploaded.items():
        Path(name).write_bytes(data)
        if name.endswith('.zip'):
            import zipfile
            with zipfile.ZipFile(name) as z:
                z.extractall('.')

# ── Build submission zip ──────────────────────────────────────────────────
print('\n── Running bundle.py ──')
result = subprocess.run(
    ['python3', 'bundle.py', 'homework', UT_ID],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('BUNDLE ERROR:\n', result.stderr)
    raise RuntimeError('bundle.py failed — fix the error above before downloading')

zip_path = Path(f'{UT_ID}.zip')
if not zip_path.exists():
    raise FileNotFoundError(f'{zip_path} was not created — check bundle.py output above')

print(f'Zip size: {zip_path.stat().st_size / 1e6:.1f} MB')

# ── Verify with the local grader ──────────────────────────────────────────
print('\n── Running grader ──')
result = subprocess.run(
    ['python3', '-m', 'grader', str(zip_path)],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('GRADER ERROR:\n', result.stderr)

# ── Download to PC ────────────────────────────────────────────────────────
files.download(str(zip_path))

In [ ]:
import shutil
from pathlib import Path
from google.colab import drive, files

UT_ID = 'ja56388'   # ← must match the UT EID used in the submission cell
zip_path = Path(f'{UT_ID}.zip')

if not zip_path.exists():
    raise FileNotFoundError(f'{zip_path} not found — run the submission cell first')

# ── Option A: Save to Google Drive ────────────────────────────────────────
drive.mount('/drive', force_remount=True)
drive_dest = Path(f'/drive/MyDrive/{UT_ID}.zip')
shutil.copy(zip_path, drive_dest)
print(f'Saved to Google Drive: {drive_dest}')

# ── Option B: Download directly to your PC ────────────────────────────────
files.download(str(zip_path))
print('Browser download triggered.')